Packages needed

In [1]:
import tensorflow as tf
import pandas as pd
import numpy as np
import keras
import tensorflow_addons as tfa
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from transformers import RobertaTokenizer, TFRobertaForSequenceClassification, create_optimizer
from tensorflow.keras.callbacks import Callback, EarlyStopping
from tensorflow.keras.optimizers import Adam


C:\Users\ronan\anaconda3\envs\nlp_env\lib\site-packages\tensorflow_addons\utils\tfa_eol_msg.py:23: UserWarning: 

TensorFlow Addons (TFA) has ended development and introduction of new features.
TFA has entered a minimal maintenance and release mode until a planned end of life in May 2024.
Please modify downstream libraries to take dependencies from other repositories in our TensorFlow community (e.g. Keras, Keras-CV, and Keras-NLP). 

For more information see: https://github.com/tensorflow/addons/issues/2807 

  warnings.warn(
C:\Users\ronan\anaconda3\envs\nlp_env\lib\site-packages\tensorflow_addons\utils\ensure_tf_install.py:53: UserWarning: Tensorflow Addons supports using Python ops for all Tensorflow versions above or equal to 2.12.0 and strictly below 2.15.0 (nightly versions are not supported). 
 The versions of TensorFlow you are currently using is 2.11.0 and is not supported. 
Some things might work, some things might not.
If you were to encounter a bug, do not file an issue.
I

Loading Datasets

In [2]:
TrainingDataset = pd.read_json('C:/Users/ronan/sandbox/NLP/NLPassignmentOne/subtaskA_train_monolingual.jsonl', lines=True)
TrainingDataset = TrainingDataset[['text', 'label']]

TestDataset = pd.read_json('C:/Users/ronan/sandbox/NLP/NLPassignmentOne/subtaskA_dev_monolingual.jsonl', lines=True)
TestDataset = TestDataset[['text', 'label']]

Tokenize data

In [3]:
tkizer = RobertaTokenizer.from_pretrained('roberta-base')

C:\Users\ronan\anaconda3\envs\nlp_env\lib\site-packages\huggingface_hub\file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizing text

In [4]:
length = 128

TrainEncodings = tkizer(list(TrainingDataset['text']), truncation=True, padding='max_length', max_length=length)
TestEncodings = tkizer(list(TestDataset['text']), truncation=True, padding='max_length', max_length=length)

labels

In [5]:
TrainLabels = list(TrainingDataset['label'])
TestLabels = list(TestDataset['label'])

In [6]:
BatchSize = 32

TrainDataset = tf.data.Dataset.from_tensor_slices((dict(TrainEncodings), TrainLabels))
TrainDataset = TrainDataset.shuffle(len(TrainLabels)).batch(BatchSize)

TestDataset = tf.data.Dataset.from_tensor_slices((dict(TestEncodings), TestLabels))
TestDataset = TestDataset.shuffle(len(TestLabels)).batch(BatchSize)

In [7]:
model = TFRobertaForSequenceClassification.from_pretrained('roberta-base', num_labels = 2)

All model checkpoint layers were used when initializing TFRobertaForSequenceClassification.

Some layers of TFRobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [8]:
loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

In [9]:
metrics = ['accuracy']

In [10]:
epochs = 3
steps_per_epoch = len(TrainLabels) // BatchSize
num_train_steps = steps_per_epoch * epochs

optimizer, lr_schedule  = create_optimizer(
    init_lr=2e-5,
    num_train_steps=num_train_steps,
    num_warmup_steps=0
)

In [11]:
model.compile(optimizer=optimizer, loss=loss, metrics = metrics)

In [12]:
model.fit(TrainDataset, validation_data = TestDataset, epochs=1)

3743/3743 [==============================] - 35212s 9s/step - loss: 0.0649 - accuracy: 0.9754 - val_loss: 1.6564 - val_accuracy: 0.7006


In [15]:
yPredictions = model.predict(TestDataset)
yLabels = np.argmax(yPredictions.logits, axis=1)

yTrue = []
for batch in TestDataset:
    _, labels = batch
    yTrue.extend(labels.numpy())
yTrue = np.array(yTrue)

ValidateF1 = f1_score(yTrue, yLabels, average='macro')
acc = accuracy_score(yTrue, yLabels)

157/157 [==============================] - 342s 2s/step


In [16]:
print(f"Validation Accuracy: {acc:.4f}")
print(f"Validation F1 Score: {ValidateF1:.4f}")

Validation Accuracy: 0.5000
Validation F1 Score: 0.3333
